# Multi-Agent Codebase Management with Hugging Face Inference + LangGraph
## Analyzer → Improver → Reviewer collaborating on code quality

Agents work together on a code snippet or file:
- **Analyzer** — identifies issues, complexity, missing tests, style problems
- **Improver** — proposes refactors, fixes, or better implementations
- **Reviewer** — scores the change and decides next step (accept, revise, re-analyze)

Hard max iterations + shared state keep the loop safe and traceable.

In [ ]:
import os
import json
import re
from typing import TypedDict, List
from huggingface_hub import InferenceClient
from langgraph.graph import StateGraph, END

HF_TOKEN = os.getenv("HF_TOKEN", "hf_your_token_here")
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # or a code-specialized model

client = InferenceClient(token=HF_TOKEN)
print(f"Using model: {MODEL}")

In [ ]:
class State(TypedDict):
    goal: str
    code: str
    analysis: str
    improved_code: str
    feedback: str
    score: float
    attempts: int
    max_attempts: int
    history: List[dict]
    next_action: str  # "analyze" | "improve" | "accept" | "max"

In [ ]:
def call_llm(system: str, user: str, max_tokens: int = 800, temperature: float = 0.4) -> str:
    response = client.chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()


def analyzer_node(state: State) -> dict:
    print("\n--- Analyzer ---")
    prompt = f"""Analyze this code against the goal.

Goal: {state['goal']}

Code:
```python
{state['code']}
```

Previous feedback:
{state.get('feedback', 'None')}

Report:
- Bugs or correctness issues
- Complexity / readability problems
- Missing error handling or edge cases
- Opportunities for improvement
- Suggested test cases"""
    analysis = call_llm(
        "You are a precise code analyst. Be concrete and actionable.",
        prompt,
        temperature=0.3,
    )
    print(analysis[:500] + "...")
    return {"analysis": analysis}


def improver_node(state: State) -> dict:
    print("\n--- Improver ---")
    prompt = f"""Improve the code based on the analysis and feedback.

Goal: {state['goal']}

Analysis:
{state['analysis']}

Current code:
```python
{state['code']}
```

Critic feedback:
{state.get('feedback', 'None')}

Return ONLY the improved Python code inside a single ```python block.
Keep the same function signatures unless the analysis explicitly requires changes."""
    raw = call_llm(
        "You are a skilled code improver. Output clean, correct Python.",
        prompt,
        max_tokens=1000,
        temperature=0.5,
    )
    # Extract code block if present
    match = re.search(r"```python\n(.*?)```", raw, re.DOTALL)
    improved = match.group(1).strip() if match else raw.strip()
    print(improved[:400] + "...")
    return {
        "improved_code": improved,
        "code": improved,  # update working code
        "attempts": state["attempts"] + 1,
    }


def reviewer_node(state: State) -> dict:
    print("\n--- Reviewer ---")
    prompt = f"""Review the improved code.

Goal: {state['goal']}
Analysis:
{state['analysis']}

Improved code:
```python
{state['improved_code']}
```

Score 1-10 for correctness, clarity, and completeness.
Decide next_action:
- "accept" if score >= 8 and the code meets the goal
- "improve" if more polishing is needed
- "analyze" if deeper issues were missed

Respond ONLY with valid JSON:
{{
  "score": 7.5,
  "next_action": "improve",
  "feedback": "Specific feedback for the next agent."
}}"""
    raw = call_llm(
        "You are a rigorous code reviewer. Output only JSON.",
        prompt,
        max_tokens=400,
        temperature=0.2,
    )
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    try:
        result = json.loads(match.group()) if match else {}
    except Exception:
        result = {}

    score = float(result.get("score", 5.0))
    next_action = result.get("next_action", "improve")
    feedback = result.get("feedback", raw[:250])

    if score >= 8.0:
        next_action = "accept"
    if state["attempts"] >= state["max_attempts"]:
        next_action = "max"

    history = state.get("history", []) + [{
        "attempt": state["attempts"],
        "score": score,
        "next_action": next_action,
        "feedback": feedback,
    }]

    print(f"Score: {score} | Next: {next_action}")
    print(f"Feedback: {feedback}")

    return {
        "score": score,
        "feedback": feedback,
        "next_action": next_action,
        "history": history,
    }


def router(state: State) -> str:
    action = state.get("next_action", "improve")
    if action in ("accept", "max"):
        return END
    if action == "analyze"::
        return "analyzer"
    return "improver"

In [ ]:
# Build the graph
workflow = StateGraph(State)
workflow.add_node("analyzer", analyzer_node)
workflow.add_node("improver", improver_node)
workflow.add_node("reviewer", reviewer_node)

workflow.set_entry_point("analyzer")
workflow.add_edge("analyzer", "improver")
workflow.add_edge("improver", "reviewer")

workflow.add_conditional_edges(
    "reviewer",
    router,
    {
        "analyzer": "analyzer",
        "improver": "improver",
        END: END,
    },
)

app = workflow.compile()
print("Codebase management graph ready.")

In [ ]:
# Example: improve a simple but flawed function
GOAL = "Write a robust function that computes the factorial of a non-negative integer, with clear error handling and type hints."

STARTING_CODE = '''
def factorial(n):
    result = 1
    for i in range(1, n+1):
        result *= i
    return result
'''

initial = {
    "goal": GOAL,
    "code": STARTING_CODE.strip(),
    "analysis": "",
    "improved_code": "",
    "feedback": "",
    "score": 0.0,
    "attempts": 0,
    "max_attempts": 3,
    "history": [],
    "next_action": "analyze",
}

result = app.invoke(initial)

print("\n" + "="*70)
print("FINAL IMPROVED CODE")
print("="*70)
print(result["improved_code"])
print("\n--- History ---")
for h in result["history"]:
    print(f"Attempt {h['attempt']}: score={h['score']} → {h['next_action']}")

## How the agents manage the codebase together

1. **Analyzer** inspects the current code against the goal.
2. **Improver** produces a better version.
3. **Reviewer** scores it and routes:
   - Accept → done
   - Back to Improver for polish
   - Back to Analyzer if deeper issues remain
4. Hard attempt limit; full history is kept.

### Real-world extensions
- Point the Analyzer at a whole file or repo diff (git diff)
- Add a **Tester** agent that generates and runs unit tests
- Add a **Documenter** agent for docstrings / README updates
- Use tool calling so agents can run `pytest`, `ruff`, or `mypy`
- Checkpoint the graph so long reviews can pause and resume
- Human approval gate before accepting changes

This pattern scales from single functions to multi-file refactors and PR review loops.